<a href="https://colab.research.google.com/github/Aashutosh2021/openwakeword_tarining/blob/main/notebooks/custom_human_voice_training_fixed_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenWakeWord: Custom Human Voice Training
This notebook is optimized for training a wake word model primarily on your **real voice recordings**.
It still generates a small amount of synthetic data to create "hard negatives" (words that sound like your wake word but aren't) to prevent false activations, but the vast majority of the training will focus on your actual voice!

### Step 1: Install Dependencies
Run this cell to install OpenWakeWord and download the required background noise datasets.

In [1]:
!git clone https://github.com/dscripka/openWakeWord.git
!git clone https://github.com/dscripka/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# FIX: PyTorch 2.6+ changed torch.load() default from weights_only=False to True.
# generate_samples.py calls torch.load(model_path) with no argument -> blocks loading
# the older piper checkpoint (UnpicklingError: Unsupported global SynthesizerTrn).
# Patch the source directly (trusted file, official openWakeWord/piper pipeline).
!sed -i 's/model = torch.load(model_path)/model = torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py
!pip install piper-phonemize
!pip install webrtcvad
%cd openWakeWord
!pip install -e .

# --- FIX: download required melspectrogram/embedding onnx+tflite models ---
# pip install -e . does NOT auto-download these; AudioFeatures() and train.py both need them.
# cwd is now /content/openWakeWord (after %cd above), so path is single-nested, not double.
import os
os.makedirs('openwakeword/resources/models', exist_ok=True)
!wget -O openwakeword/resources/models/embedding_model.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx
!wget -O openwakeword/resources/models/embedding_model.tflite https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite
!wget -O openwakeword/resources/models/melspectrogram.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx
!wget -O openwakeword/resources/models/melspectrogram.tflite https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite

!pip install tensorflow
!pip install pronouncing
!pip install audiomentations
!pip install torch-audiomentations
!pip install speechbrain
!pip install mutagen
!pip install acoustics
!pip install torchinfo
!pip install torchmetrics
!pip install webrtcvad
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19
!pip install espeak-phonemizer
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

Cloning into 'openWakeWord'...
remote: Enumerating objects: 1248, done.
remote: Counting objects: 100% (721/721), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 1248 (delta 602), reused 561 (delta 561), pack-reused 527 (from 1)
Receiving objects: 100% (1248/1248), 3.23 MiB | 2.69 MiB/s, done.
Resolving deltas: 100% (775/775), done.
Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 75 (delta 21), reused 18 (delta 18), pack-reused 40 (from 1)
Receiving objects: 100% (75/75), 1.01 MiB | 21.14 MiB/s, done.
Resolving deltas: 100% (22/22), done.
--2026-07-26 11:50:06--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 

### Step 2: Upload Your Voice Clips
1. Record 50-100 clips of yourself saying your wake word (16kHz, 16-bit, Mono `.wav` format).
2. Put them in a `.zip` file named `my_positive_clips.zip`.
3. Upload that `.zip` file to the Colab sidebar.
4. Run the cell below to extract them!

In [2]:
!unzip /content/my_positive_clips.zip -d /content/
print("Voice clips extracted to /content/my_positive_clips/")

Archive:  /content/my_positive_clips.zip
   creating: /content/my_positive_clips/
  inflating: /content/my_positive_clips/Recording (10).wav  
  inflating: /content/my_positive_clips/Recording (11).wav  
  inflating: /content/my_positive_clips/Recording (12).wav  
  inflating: /content/my_positive_clips/Recording (13).wav  
  inflating: /content/my_positive_clips/Recording (14).wav  
  inflating: /content/my_positive_clips/Recording (15).wav  
  inflating: /content/my_positive_clips/Recording (16).wav  
  inflating: /content/my_positive_clips/Recording (17).wav  
  inflating: /content/my_positive_clips/Recording (18).wav  
  inflating: /content/my_positive_clips/Recording (19).wav  
  inflating: /content/my_positive_clips/Recording (2).wav  
  inflating: /content/my_positive_clips/Recording (20).wav  
  inflating: /content/my_positive_clips/Recording (21).wav  
  inflating: /content/my_positive_clips/Recording (22).wav  
  inflating: /content/my_positive_clips/Recording (23).wav  
  in

### Step 3: Configure Your Wake Word
Change the `target_phrase` below to the word(s) you recorded yourself saying.

In [15]:
import yaml
import os

# Load the default config from openWakeWord to ensure no missing keys!
with open('/content/openWakeWord/examples/custom_model.yml', 'r') as f:
    config = yaml.load(f.read(), yaml.Loader)

# Override with our custom settings
config["target_phrase"] = ["ultron"] # Change this to your wake word
config["model_name"] = "custom_voice_model"
config["piper_sample_generator_path"] = "/content/piper-sample-generator"  # FIX: cloned as sibling of openWakeWord, not inside it
config["n_samples"] = 100 # Keep low so real voice dominates
config["n_samples_val"] = 100
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["background_paths"] = ['/content/openWakeWord/audioset_16k', '/content/openWakeWord/fma']  # FIX: absolute, cwd-proof
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)
print("Configuration saved successfully with all default keys intact!")

Configuration saved successfully with all default keys intact!


### Step 4: Generate Synthetic Negatives & Phonetic Variations
This runs quickly because we lowered the `n_samples`. It generates words that sound similar to your wake word to teach the model what *not* to trigger on.

In [4]:
!apt-get install -y espeak-ng

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 53 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng-data amd64 1.50+dfsg-10ubuntu0.1 [3,956 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libespeak-ng1 amd64 1.50+dfsg-10ubuntu0.1 [207 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 espeak-ng amd64 1.50+dfsg-1

In [5]:
import torchaudio
import os

# Get the path to torchaudio's __init__.py
torchaudio_init_path = os.path.join(os.path.dirname(torchaudio.__file__), "__init__.py")

# Append a dummy function to bypass the error
with open(torchaudio_init_path, "a") as f:
    f.write("\ndef set_audio_backend(backend):\n")
    f.write("    pass\n")

print("Torchaudio successfully patched!")

Torchaudio successfully patched!


In [6]:
import os
import scipy.io.wavfile
import numpy as np
from tqdm import tqdm
import datasets

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

print('Downloading Room Impulse Responses (for augmentation)...')
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print('Finished downloading RIRs!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [03:35,  1.25it/s]

Finished downloading RIRs!


In [16]:
import os
import scipy.io.wavfile
import numpy as np
from tqdm import tqdm
import datasets
from pathlib import Path

print('Downloading Audioset background noise...')
output_dir = "/content/openWakeWord/audioset_16k"  # FIX: absolute, cwd-proof
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

!pip install -q soundfile

import io
import pyarrow.parquet as pq
import soundfile as sf
import torch
import torchaudio

# FIX 4: datasets==2.14.6's builder/download-manager pipeline itself is broken under
# this Colab's Python 3.12. Bypass `datasets` entirely: read parquet with pyarrow,
# decode embedded FLAC bytes with soundfile, resample with torchaudio, write wav ourselves.
os.makedirs("audioset_parquet", exist_ok=True)
shard_files = ["00.parquet", "01.parquet", "02.parquet", "03.parquet", "04.parquet"]
for shard in shard_files:
    url = f"https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/{shard}"
    out = f"audioset_parquet/{shard}"
    if not os.path.exists(out):
        !wget -q -O {out} {url}

n_clips = 1800
count = 0
for shard in shard_files:
    if count >= n_clips:
        break
    table = pq.read_table(f"audioset_parquet/{shard}", columns=["audio"])
    audio_col = table.column("audio").to_pylist()
    for item in tqdm(audio_col, desc=shard):
        if count >= n_clips:
            break
        try:
            data, sr = sf.read(io.BytesIO(item["bytes"]), dtype="float32")
        except Exception:
            continue
        if data.ndim > 1:
            data = data.mean(axis=1)
        if sr != 16000:
            wav = torchaudio.functional.resample(torch.from_numpy(data).unsqueeze(0), sr, 16000)
            data = wav.squeeze(0).numpy()
        fname = os.path.basename(item["path"]).replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, fname), 16000, (data * 32767).astype(np.int16))
        count += 1
print(f"Saved {count} Audioset clips.")

print('Downloading FMA background noise...')
output_dir = "/content/openWakeWord/fma"  # FIX: absolute, cwd-proof
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1
for i in tqdm(range(n_hours*3600//30)):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break
print('Finished downloading background noise!')

03.parquet:  60%|██████    | 300/500 [00:09<00:06, 32.18it/s]


Saved 1800 Audioset clips.


 99%|█████████▉| 119/120 [00:40<00:00,  2.97it/s]

Finished downloading background noise!


In [13]:
# NOTE: this file is REQUIRED, not optional.
# generate_samples() in piper-sample-generator defaults its `model=` arg to
# <piper_sample_generator_path>/models/en-us-libritts-high.pt, and openWakeWord's
# train.py never overrides that default. Without this exact file at this exact
# path, --generate_clips fails at torch.load() with FileNotFoundError.
!wget -O /content/piper-sample-generator/models/en-us-libritts-high.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'


--2026-07-26 12:13:25--  https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/40dc6d54-317d-4987-a114-8f2a02c04126?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-26T13%3A01%3A01Z&rscd=attachment%3B+filename%3Den-us-libritts-high.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-26T12%3A01%3A01Z&ske=2026-07-26T13%3A01%3A01Z&sks=b&skv=2018-11-09&sig=7yX3qxx2vS739jJpyC8%2BuuFfi5eCcJMjXO0kR%2FfX6pg%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4NTA3MTYwNiwibmJmIjoxNzg1MDY4MDA2LCJwYXRoIjoic

In [18]:
!pip install torchaudio
!pip install torchinfo
!pip install torchmetrics

In [20]:
# FIX 5: whack-a-mole ender. PyTorch 2.6+ defaults torch.load(weights_only=True),
# breaking every legacy checkpoint loader in this pipeline (generate_samples.py,
# deep-phonemizer's dp/model/model.py, and possibly more). Instead of sed-patching
# each library file one at a time, monkeypatch torch.load globally via sitecustomize.py
# -> Python auto-imports this at startup for EVERY process, including the train.py
# subprocess, before any library code runs.
import site, os
sitecustomize_path = os.path.join(site.getsitepackages()[0], "sitecustomize.py")
with open(sitecustomize_path, "w") as f:
    f.write(
        "import torch\n"
        "_orig_load = torch.load\n"
        "def _patched_load(*args, **kwargs):\n"
        "    kwargs.setdefault('weights_only', False)\n"
        "    return _orig_load(*args, **kwargs)\n"
        "torch.load = _patched_load\n"
    )
print(f"sitecustomize.py written to {sitecustomize_path}")

sitecustomize.py written to /usr/local/lib/python3.12/dist-packages/sitecustomize.py


In [31]:
!sed -i 's/checkpoint = torch.load(checkpoint_path, map_location=device)/checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)/' /usr/local/lib/python3.12/dist-packages/dp/model/model.py

In [33]:
import torchaudio
import os

# Get the path to torchaudio's __init__.py
torchaudio_init_path = os.path.join(os.path.dirname(torchaudio.__file__), "__init__.py")

# Append a dummy function to bypass the error
with open(torchaudio_init_path, "a") as f:
    f.write("\ndef set_audio_backend(backend):\n")
    f.write("    pass\n")

print("Torchaudio successfully patched!")

Torchaudio successfully patched!


In [34]:
import sys
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --generate_clips
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --augment_clips

INFO:root:##################################################
Generating positive clips for training
##################################################
INFO:root:##################################################
Generating positive clips for testing
##################################################
INFO:root:##################################################
Generating negative clips for training
##################################################
INFO:root:##################################################
Generating negative clips for testing
##################################################


### Step 5: Inject Your Real Human Voice Data
This cell converts your `.wav` files into OpenWakeWord features and seamlessly merges them into the training dataset.

In [26]:
import os
import glob
import numpy as np
import scipy.io.wavfile
import torch
import torchaudio
from openwakeword.utils import AudioFeatures

custom_clips_dir = "/content/my_positive_clips"
TARGET_LEN = 32000  # 2.0s @ 16kHz - matches train.py's default/most-common total_length

def load_and_pad(path, target_len=TARGET_LEN):
    sr, data = scipy.io.wavfile.read(path)
    data = data.astype(np.float32)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != 16000:
        wav = torchaudio.functional.resample(torch.from_numpy(data).unsqueeze(0), sr, 16000)
        data = wav.squeeze(0).numpy()
    if len(data) < target_len:
        data = np.pad(data, (0, target_len - len(data)))
    else:
        data = data[:target_len]
    return data.astype(np.int16)

def match_frames(feat, target_frames):
    """Pad/truncate the frame dimension so vstack with existing features never
    crashes, regardless of whether our TARGET_LEN guess matches train.py's actual
    internal total_length for this run."""
    n = feat.shape[1]
    if n == target_frames:
        return feat
    if n > target_frames:
        return feat[:, :target_frames, :]
    pad = np.repeat(feat[:, -1:, :], target_frames - n, axis=1)
    return np.concatenate([feat, pad], axis=1)

if os.path.exists(custom_clips_dir):
    positive_clips = glob.glob(os.path.join(custom_clips_dir, "*.wav"))
    if len(positive_clips) > 0:
        print(f"Found {len(positive_clips)} custom positive clips. Extracting features...")
        F = AudioFeatures(device="cpu")

        # FIX: embed_clips() needs a numpy array of raw audio, shape (N, samples),
        # all same length - not a list of file path strings. Load + pad/truncate first.
        clips_array = np.stack([load_and_pad(p) for p in positive_clips])
        custom_features = F.embed_clips(clips_array, batch_size=16)

        if isinstance(custom_features, dict):
            custom_features = list(custom_features.values())[0]
        if isinstance(custom_features, list):
            custom_features = np.vstack(custom_features)

        model_name = config["model_name"]
        output_dir = os.path.join(config.get("output_dir", "my_custom_model"), model_name)
        train_path = os.path.join(output_dir, "positive_features_train.npy")
        val_path = os.path.join(output_dir, "positive_features_val.npy")

        if os.path.exists(train_path):
            existing_train = np.load(train_path)
            custom_matched = match_frames(custom_features, existing_train.shape[1])
            weight_multiplier = 4
            # FIX: was np.tile(custom_features, (4, 1)) - wrong for a 3D array,
            # it tiled the FRAMES axis instead of the CLIPS axis. Now (4, 1, 1).
            weighted_custom = np.tile(custom_matched, (weight_multiplier, 1, 1))
            new_train = np.vstack([existing_train, weighted_custom])
            np.save(train_path, new_train)
            print(f"Appended weighted real clips to training features. New total: {new_train.shape[0]}")

        if os.path.exists(val_path):
            existing_val = np.load(val_path)
            custom_matched = match_frames(custom_features, existing_val.shape[1])
            val_features = custom_matched[:max(1, len(custom_matched)//5)]
            new_val = np.vstack([existing_val, val_features])
            np.save(val_path, new_val)
            print(f"Appended real clips to validation features. New total: {new_val.shape[0]}")
    else:
        print(f"Directory {custom_clips_dir} exists, but no .wav files found.")
else:
    print(f"No custom real clips found at {custom_clips_dir}. Skipping...")

Found 26 custom positive clips. Extracting features...
Appended weighted real clips to training features. New total: 308


### Step 6: Train the Model!
Train the final neural network. Once complete, download the `.onnx` file from `/content/openWakeWord/my_custom_model/custom_voice_model/`.

In [ ]:
!rm -rf /content/openWakeWord/my_custom_model/custom_voice_model

In [43]:
import sys
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --generate_clips
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --augment_clips

INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:generate_samples:Loading /content/piper-sample-generator/models/en-us-libritts-high.pt
INFO:generate_samples:Successfully loaded the model
DEBUG:generate_samples:CUDA available, using GPU
/content/piper-sample-generator/generate_samples.py:113: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  resampler = torchaudio.transforms.Resample(
DEBUG:generate_samples:Batch 1/2 complete
DEBUG:generate_samples:Batch 2/2 complete
INFO:generate_samples:Done
INFO:root:##################################################
Generating positive clips for testing
##################################################
DEBUG:generate_samples:Loading /content/piper-sample-generator/models/en-us-libritts-high.pt
INFO:generate_samples:Succes

In [44]:
!cp -r /content/positive_features_test.npy /content/openWakeWord/my_custom_model/custom_voice_model
!cp -r /content/negative_features_test.npy /content/openWakeWord/my_custom_model/custom_voice_model
!cp -r /content/negative_features_train.npy /content/openWakeWord/my_custom_model/custom_voice_model

In [45]:
import sys
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --train_model

INFO:root:##################################################
Starting training sequence 1...
##################################################
Training: 100% 9999/10000 [06:00<00:00, 27.71it/s]
INFO:root:##################################################
Starting training sequence 2...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...
Training: 100% 999/1000.0 [02:47<00:00,  5.97it/s]
INFO:root:##################################################
Starting training sequence 3...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...
Training:  95% 947/1000.0 [18:31<01:02,  1.17s/it]
Traceback (most recent call last):
  File "/content/openWakeWord/openwakeword/train.py", line 895, in <module>
    best_model = oww.auto_train(
                 ^^^^^^^^^^^^^^^
  File "/content/openWakeWord/openwakeword/train.py", line 317, in auto_train
    